In [ ]:
import numpy as np
from scipy.signal import istft

import matplotlib.pyplot as plt

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import LoRa, MultiBAMv3

import importlib
import utils.my_lora_utils
importlib.reload(utils.my_lora_utils)
from utils.my_lora_utils import *
import torch
print(utils.my_lora_utils.__file__)

c:\Users\priba\Sean-2025\INC-LAB\BAM\INC-BAM\utils\my_lora_utils.py


In [ ]:
sf = 9
bw = 125000
fs = 1000000
lora_init = LoRa(sf, bw)
folder_real = "weight_test_real"
folder_im = "weight_test_im"
layers = [256 * 17,2048,512] 
## HOW TO LOAD WEIGHTweight_image_4352_2048_2048_512
multi_bam_real = MultiBAMv3(layers_dims=layers, eta=1e-5)
for i, bam in enumerate(multi_bam_real.bams):
    w_np = np.load(f"{folder_real}/weights_layer_{i}.npy")
    bam.W = torch.tensor(w_np, dtype=torch.float32, device=bam.device)
    print(f"Loaded layer {i}: shape {bam.W.shape}")

multi_bam_im = MultiBAMv3(layers_dims=layers, eta=1e-5)
for i, bam_ in enumerate(multi_bam_im.bams):
    w_np = np.load(f"{folder_im}/weights_layer_{i}.npy")
    bam.W = torch.tensor(w_np, dtype=torch.float32, device=bam.device)
    print(f"Loaded layer {i}: shape {bam_.W.shape}")
# for i, bam_ in enumerate(multi_bam_real.bams):
#     print(f"Layer {i} weights:\n", bam_.W)
#     print("Shape:", bam_.W.shape)
#     print("-" * 40)

# for i, bam_ in enumerate(multi_bam_im.bams):
#     print(f"Layer {i} weights:\n", bam_.W)
#     print("Shape:", bam_.W.shape)
#     print("-" * 40)


In [ ]:
# EXAMPLE OF COMPRESS and DECOMPRESS
x1 = lora_init.gen_symbol_fs(300, sf=sf, bw=bw, Fs=int(bw*8))  # you had Fs=int(bw*8)=1e6
snr = 10
x = lora_init.awgn_iq(x1,snr)

#### STEP 1 PREPROCESS : DOWNSAMPLING ########
x_ds,fs_new = downsampling(x,fs,4)                                                      

PLOT_SPECGRAM(x_ds,128,"a",32,fs_new)

#### STEP 2 Create Spectrogram ########
real_comp,imag_comp,r_min,r_max,i_min,i_max = create_spectrogram_npy_dual(x_ds,fs_new,snr,256,1,folder_r=None,folder_i=None)
real_inverse_norm = real_comp * (r_max - r_min) + r_min
imag_inverse_norm = imag_comp * (i_max - i_min) + i_min
complex_b = real_inverse_norm + 1j * imag_inverse_norm
show_multiple_spectrograms([real_comp,imag_comp,np.abs(complex_b)],["real original","ima original","complex view"],3)
#### STEP 3 FLATTEN INPUT ########
flat_real = real_comp.flatten().reshape(1, -1) # BEFORE COMPRESS MUST 
flat_imag = imag_comp.flatten().reshape(1, -1) # BEFORE COMPRESS MUST 
print(flat_real.shape)
out_real = multi_bam_real.compress(flat_real)
out_im = multi_bam_im.compress(flat_imag)

decompress_real = multi_bam_real.decompress(out_real)
decompress_imag = multi_bam_im.decompress(out_im)

#### STEP 4 AFTER DECOMPRESS RESHAPE to INITIAL SHAPE ########
reco_flat_r = decompress_real.reshape(-1)  # AFTER DECOMPRESS MUST
reco_spec_r = reco_flat_r.reshape(256, 17)
check_simmilarity(reco_spec_r,real_comp)
reco_flat_i = decompress_imag.reshape(-1)  # AFTER DECOMPRESS MUST
reco_spec_i = reco_flat_i.reshape(256, 17)

# real_inverse_norm =((reco_spec_r + 1) / 2) * (r_max - r_min) + r_min #max , min , min
# imag_inverse_norm =((reco_spec_i + 1) / 2) * (i_max - i_min) + i_min
real_inverse_norm = reco_spec_r * (r_max - r_min) + r_min
imag_inverse_norm = reco_spec_i * (i_max - i_min) + i_min

# real_inverse_norm =reco_spec_r 
# imag_inverse_norm =reco_spec_i
complex_b = real_inverse_norm + 1j * imag_inverse_norm
show_multiple_spectrograms([real_inverse_norm,imag_inverse_norm,np.abs(complex_b)],["real after decompress","ima after decompress","complex view"],3)
Zxx_complex = real_inverse_norm + 1j * imag_inverse_norm

nperseg = 128 #128
noverlap = 64 # 64
nfft = 512 #512 

Zxx_unshifted = np.fft.ifftshift(Zxx_complex, axes=0)
# PROCESS ISTF
_, x_rec = istft(
    Zxx_unshifted, fs=fs_new, window="hann", nperseg=nperseg,
    noverlap=noverlap, nfft=nfft, input_onesided=False
)

sym,max = estimate_symbol_custom(x_rec,"a",9,250000,125000,tresshold = 0)
print(sym)
print(max)
PLOT_SPECGRAM(x_rec,128,"After decompress",32,fs_new)

In [ ]:
sf = 9
bw = 125000
fs = 1000000
lora_init = LoRa(sf, bw)
import torch
## HOW TO LOAD WEIGHT
layers = [256*15, 1024, 256] # <-- must match training

multi_bam = MultiBAMv3(layers_dims=layers, eta=1e-5)

for i, bam in enumerate(multi_bam.bams):
    w_np = np.load(f"weight_test/weights_layer_{i}.npy")
    bam.W = torch.tensor(w_np, dtype=torch.float32, device=bam.device)

symbol1 = 0
symbol2 = 0
# EXAMPLE OF COMPRESS and DECOMPRESS

x1 = lora_init.gen_symbol_fs(symbol1, sf=sf, bw=bw, Fs=int(bw*8))  # you had Fs=int(bw*8)=1e6
snr = -22
x = lora_init.awgn_iq(x1,snr)
x_down,fs_new_down = downsampling(x,fs,4)
image_ori_noise,null,null = create_spectrogram_npy(x_down,fs_new_down,0,0,1,None)

x2 = apply_cfo(x, Fs=fs_new_down, freq_offset_hz=300)
x2 = apply_phase_noise(x2, Fs=fs_new_down, linewidth_hz=50)
x2 = multipath_rayleigh(x2, [0, 5], 6)
x2 = band_limited_noise(x2, Fs=fs_new_down, low_hz=110e3, high_hz=120e3, snr_db=20)
x2 = quantize_iq(x2, nbits=8)
x2 = time_varying_rayleigh(x2,10,fs_new_down)
x2 =  hard_clip(x2, max_amp=0.1)

#### STEP 1 PREPROCESS : DOWNSAMPLING ########
x_ds,fs_new = downsampling(x2,fs,4)

#### STEP 2 Create Spectrogram ########
aa,bb,cc = create_spectrogram_npy(x_ds,fs_new,0,0,1,None)

#### STEP 3 FLATTEN INPUT ########
flat = aa.flatten().reshape(1, -1) # BEFORE COMPRESS MUST 
out1 = multi_bam.compress(flat)
print(out1.shape)
out2 = multi_bam.decompress(out1)
print(out2.shape)

#### STEP 4 AFTER DECOMPRESS RESHAPE to INITIAL SHAPE ########
reco_flat = out2.reshape(-1)  # AFTER DECOMPRESS MUST
reco_spec = reco_flat.reshape(256, 15)
print(reco_spec.shape)
array_a = [image_ori_noise,aa,reco_spec]

show_multiple_spectrograms(array_a,["ori noise","broken","decompress"],3)